# 从零实现 BPE：训练、编码与工程边界

## 学习目标

完成本 notebook 后，你应该能够：

1. 区分 BPE 的训练、编码和解码三个阶段；
2. 正确统计带词频权重的相邻 pair，并处理重叠替换；
3. 解释为什么推理使用固定 merge rank，而不是重新统计输入；
4. 独立实现一个可运行的字符级 BPE，并分析 OOV、复杂度和工程取舍。

> 这里使用字符加词尾标记的教学版本。现代 LLM 常使用 byte-level BPE，但合并与 rank 的核心思想一致。


## 1. 原理与公式

设词 $w$ 当前表示为符号序列 $s_w=(s_1,\ldots,s_n)$，词频为 $f(w)$。相邻 pair $(a,b)$ 的加权频次为：

\[
C(a,b)=\sum_w f(w)\sum_{i=1}^{|s_w|-1}\mathbf{1}[s_i=a\land s_{i+1}=b].
\]

每轮选择 $\arg\max C(a,b)$，创建新符号 $ab$，并在所有序列中从左到右替换不重叠的出现。重复若干轮即可得到有序 merge 表。注意：频次相同的 pair 必须使用稳定 tie-break，否则同一语料可能得到不同词表。


In [ ]:
from collections import Counter  # 导入本单元所需的依赖。

EOW = "</w>"  # 计算并保存当前步骤的中间状态。
corpus = "low low low lower lower newest widest"  # 计算并保存当前步骤的中间状态。
word_frequency = Counter(corpus.split())  # 计算并保存当前步骤的中间状态。

# 用 tuple 作为不可变的当前分词；相同词只存一次，再乘词频。
initial_sequences = {  # 计算并保存当前步骤的中间状态。
    tuple(list(word) + [EOW]): frequency  # 执行当前语句以推进本节示例。
    for word, frequency in word_frequency.items()  # 遍历输入元素以累积或检查结果。
}  # 执行当前语句以推进本节示例。

def count_pairs(sequences):  # 定义本节可复用的核心函数。
    counts = Counter()  # 计算并保存当前步骤的中间状态。
    for symbols, frequency in sequences.items():  # 遍历输入元素以累积或检查结果。
        for left, right in zip(symbols, symbols[1:]):  # 遍历输入元素以累积或检查结果。
            counts[(left, right)] += frequency  # 计算并保存当前步骤的中间状态。
    return counts  # 返回当前分支计算出的结果。

print("词频:", word_frequency)  # 执行当前语句以推进本节示例。
print("第一轮最高频 pair:", count_pairs(initial_sequences).most_common(5))  # 执行当前语句以推进本节示例。


## 2. 不重叠合并

序列 `A A A` 中 `(A,A)` 虽有两个相邻位置，但它们共享中间元素。本轮从左到右合并只能得到 `AA A`，不能同时得到两个 `AA`。实现时匹配成功后指针前进 2，否则前进 1。

训练器下方使用 `(-频次, left, right)` 排序，既选最高频 pair，又让并列结果稳定。真实系统可选择其他 tie-break，但训练和复现必须一致。


In [ ]:
def merge_pair(symbols, pair, new_symbol):  # 定义本节可复用的核心函数。
    output = []  # 计算并保存当前步骤的中间状态。
    i = 0  # 计算并保存当前步骤的中间状态。
    while i < len(symbols):  # 在终止条件满足前持续推进状态。
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:  # 按当前条件选择后续控制路径。
            output.append(new_symbol)  # 执行当前语句以推进本节示例。
            i += 2  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            output.append(symbols[i])  # 执行当前语句以推进本节示例。
            i += 1  # 计算并保存当前步骤的中间状态。
    return tuple(output)  # 返回当前分支计算出的结果。

def train_bpe(sequences, num_merges=10, min_frequency=1):  # 定义本节可复用的核心函数。
    sequences = dict(sequences)  # 计算并保存当前步骤的中间状态。
    merges = []  # 计算并保存当前步骤的中间状态。
    for rank in range(num_merges):  # 遍历输入元素以累积或检查结果。
        counts = count_pairs(sequences)  # 计算并保存当前步骤的中间状态。
        if not counts:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        pair, frequency = min(counts.items(), key=lambda item: (-item[1], item[0]))  # 计算并保存当前步骤的中间状态。
        if frequency < min_frequency:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        new_symbol = "".join(pair)  # 计算并保存当前步骤的中间状态。
        merges.append((pair, new_symbol, frequency))  # 执行当前语句以推进本节示例。
        sequences = {  # 计算并保存当前步骤的中间状态。
            merge_pair(symbols, pair, new_symbol): word_frequency  # 执行当前语句以推进本节示例。
            for symbols, word_frequency in sequences.items()  # 遍历输入元素以累积或检查结果。
        }  # 执行当前语句以推进本节示例。
        print(f"rank={rank:02d}  {pair} -> {new_symbol!r}  count={frequency}")  # 计算并保存当前步骤的中间状态。
    return merges, sequences  # 返回当前分支计算出的结果。

merges, trained_sequences = train_bpe(initial_sequences, num_merges=10)  # 计算并保存当前步骤的中间状态。


## 3. 编码新文本：只使用训练好的 merge 顺序

编码新词时不能根据这个词重新选择最高频 pair。我们从字符和 `</w>` 开始，按训练 rank 依次应用规则。较晚规则依赖较早规则产生的符号，因此 merge 表的顺序是模型的一部分。

这个教学实现依次扫描全部规则，便于理解；生产实现通常用 pair rank、优先队列、链表和词级缓存减少重复扫描。


In [ ]:
def encode_word(word, merges):  # 定义本节可复用的核心函数。
    symbols = tuple(list(word) + [EOW])  # 计算并保存当前步骤的中间状态。
    for pair, new_symbol, _ in merges:  # 遍历输入元素以累积或检查结果。
        symbols = merge_pair(symbols, pair, new_symbol)  # 计算并保存当前步骤的中间状态。
    return list(symbols)  # 返回当前分支计算出的结果。

def encode_text(text, merges):  # 定义本节可复用的核心函数。
    return [piece for word in text.split() for piece in encode_word(word, merges)]  # 返回当前分支计算出的结果。

def decode_pieces(pieces):  # 定义本节可复用的核心函数。
    return "".join(pieces).replace(EOW, " ").rstrip()  # 返回当前分支计算出的结果。

for sample in ["low", "lower", "lowest", "newest widest"]:  # 遍历输入元素以累积或检查结果。
    pieces = encode_text(sample, merges)  # 计算并保存当前步骤的中间状态。
    print(f"{sample!r:16} -> {pieces} -> {decode_pieces(pieces)!r}")  # 执行当前语句以推进本节示例。


## 4. 边界测试与常见误区

- **重叠 pair**：替换必须不重叠；
- **忘记乘词频**：不同词型只统计一次会学出错误 merge；
- **推理重新计数**：会让同一词随上下文改变编码；
- **字符 OOV**：本实现无法表示训练字符集之外的字符；byte-level BPE 用 256 bytes 解决覆盖；
- **最长匹配等于 BPE**：不总成立，严格 BPE 依据 merge rank；
- **可逆性**：merge 可展开，但 `split()` 已丢失连续空格和换行，所以完整管线不严格可逆。


In [ ]:
# 最小边界测试
assert merge_pair(("A", "A", "A"), ("A", "A"), "AA") == ("AA", "A")  # 用受控断言验证关键不变量。
assert decode_pieces(encode_text("low lower", merges)) == "low lower"  # 用受控断言验证关键不变量。

known_characters = {char for word in word_frequency for char in word}  # 计算并保存当前步骤的中间状态。
sample = "猫"  # 计算并保存当前步骤的中间状态。
unknown = [char for char in sample if char not in known_characters]  # 计算并保存当前步骤的中间状态。
print("字符级教学模型中的未知字符:", unknown)  # 执行当前语句以推进本节示例。
print("UTF-8 byte 兜底则可表示为:", list(sample.encode("utf-8")))  # 执行当前语句以推进本节示例。


## 5. 复杂度与相邻算法对比

朴素训练执行 $M$ 轮、每轮扫描约 $N$ 个当前符号，粗略为 $O(MN)$；生产训练器维护 pair 倒排位置与局部计数。朴素编码反复扫描可能接近 $O(L^2)$，可用优先队列和缓存优化。

| 方法 | 训练方向 | 推理核心 | 多切分 |
|---|---|---|---|
| BPE | 从小词表逐步加 merge | 按固定 merge rank 合并 | 标准版本无；可用 BPE-dropout |
| WordPiece | 按似然收益或实现近似扩词表 | 常见实现为最长匹配 | 标准版本无 |
| Unigram | 从大候选词表逐步剪枝 | lattice 上 Viterbi | 天然支持 n-best/采样 |


## 练习与面试总结

1. 将字符底座改为 UTF-8 bytes，并实现严格 byte round-trip。
2. 为训练器加入稳定 token ID、vocab 导出和断点续训。
3. 用堆和邻接链表优化编码，并与参考实现做随机对拍。
4. 分别训练 20、50、100 次 merge，比较序列长度和低频词碎片度。

**一分钟回答**：BPE 从字符或字节出发，每轮合并语料中最高频相邻 pair。训练保存有序 merges，推理只按 rank 重放，不能重新统计。它用高频长 token 和低频细粒度兜底，在词表参数与序列成本之间折中；byte-level 版本几乎无 OOV，但仍要处理预分词、可逆性、长尾膨胀和确定性工程问题。
